In [18]:
# ================================
# 1. IMPORTS
# ================================
import os
import json
import re
from typing import Optional
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ================================
# 2. LLM SETUP (Production tuned)
# ================================
llm = ChatOllama(
    model="llama3.2",
    temperature=0.1,
    timeout=60
)

# ================================
# 3. SAFE JSON PARSER (CRITICAL)
# ================================
def safe_json_parse(response: str):
    try:
        return json.loads(response)
    except:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError(f"Invalid JSON from LLM:\n{response}")

# ================================
# 4. RAG SETUP (Optimized)
# ================================
loader = TextLoader("financial_rules.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def query_policies(question: str) -> str:
    docs = retriever.invoke(question)
    return "\n".join([d.page_content for d in docs])

# ================================
# 5. STATE MODEL (Validated)
# ================================
class FinancialState(BaseModel):
    user_input: str

    income: Optional[float] = Field(default=0)
    expenses: Optional[float] = Field(default=0)
    existing_emi: Optional[float] = Field(default=0)
    savings: Optional[float] = Field(default=0)
    loan_amount: Optional[float] = Field(default=0)

    retrieved_rules: Optional[str] = None

    financial_health: Optional[str] = None
    risk_score: Optional[float] = None

    emi_ratio: Optional[float] = None
    savings_ratio: Optional[float] = None

    advisory_plan: Optional[str] = None
    final_report: Optional[str] = None

# ================================
# 6. TOOL (SAFE)
# ================================
def financial_metrics_tool(income: float, emi: float, savings: float):
    if income <= 0:
        return 0, 0
    return savings / income, emi / income

# ================================
# 7. LLM CALL WRAPPER (RETRY LOGIC)
# ================================
def call_llm(prompt: str, retries=2):
    for i in range(retries):
        try:
            response = llm.invoke(prompt).content.strip()
            if response:
                return response
        except Exception as e:
            if i == retries - 1:
                raise e
    return ""

# ================================
# 8. NODE 1 — EXTRACT
# ================================
def extract_financial_profile(state: FinancialState):
    prompt = f"""
Extract financial details.

STRICT:
- Only JSON
- No explanation

Format:
{{
"income": number,
"expenses": number,
"existing_emi": number,
"savings": number,
"loan_amount": number
}}

Input: {state.user_input}
"""

    response = call_llm(prompt)
    data = safe_json_parse(response)

    state.income = data.get("income", 0)
    state.expenses = data.get("expenses", 0)
    state.existing_emi = data.get("existing_emi", 0)
    state.savings = data.get("savings", 0)
    state.loan_amount = data.get("loan_amount", 0)

    return state

# ================================
# 9. NODE 2 — RAG
# ================================
def retrieve_financial_rules(state: FinancialState):
    query = f"financial rules for income {state.income}, emi {state.existing_emi}, savings {state.savings}"
    state.retrieved_rules = query_policies(query)
    return state

# ================================
# 10. NODE 3 — LLM EVALUATION
# ================================
def evaluate_financial_health(state: FinancialState):
    prompt = f"""
You are a financial advisor.

STRICT:
- Return JSON only

User:
Income: {state.income}
Expenses: {state.expenses}
EMI: {state.existing_emi}
Savings: {state.savings}

Rules:
{state.retrieved_rules}

Return:
{{
"financial_health": "Good | Moderate | Risky",
"risk_score": number
}}
"""

    response = call_llm(prompt)
    data = safe_json_parse(response)

    state.financial_health = data.get("financial_health", "Moderate")
    state.risk_score = data.get("risk_score", 50)

    return state

# ================================
# 11. NODE 4 — TOOL NODE
# ================================
def calculate_metrics(state: FinancialState):
    s, e = financial_metrics_tool(
        state.income,
        state.existing_emi,
        state.savings
    )
    state.savings_ratio = s
    state.emi_ratio = e
    return state

# ================================
# 12. NODE 5 — ADVISORY
# ================================
def generate_advisory_plan(state: FinancialState):
    prompt = f"""
Give financial advice.

Data:
Savings Ratio: {state.savings_ratio}
EMI Ratio: {state.emi_ratio}
Risk Score: {state.risk_score}

Rules:
{state.retrieved_rules}

Output:
Bullet points only.
"""

    state.advisory_plan = call_llm(prompt)
    return state

# ================================
# 13. NODE 6 — FINAL REPORT
# ================================
def final_report(state: FinancialState):
    prompt = f"""
Create a clean report.

Health: {state.financial_health}
Risk: {state.risk_score}
Savings Ratio: {round(state.savings_ratio*100,2)}
EMI Ratio: {round(state.emi_ratio*100,2)}

Advice:
{state.advisory_plan}
"""

    state.final_report = call_llm(prompt)
    return state

# ================================
# 14. WORKFLOW
# ================================
workflow = StateGraph(FinancialState)

workflow.add_node("extract", extract_financial_profile)
workflow.add_node("rag", retrieve_financial_rules)
workflow.add_node("evaluate", evaluate_financial_health)
workflow.add_node("metrics", calculate_metrics)
workflow.add_node("advisory", generate_advisory_plan)
workflow.add_node("report", final_report)

workflow.add_edge("extract", "rag")
workflow.add_edge("rag", "evaluate")
workflow.add_edge("evaluate", "metrics")
workflow.add_edge("metrics", "advisory")
workflow.add_edge("advisory", "report")
workflow.add_edge("report", END)

workflow.set_entry_point("extract")

app = workflow.compile()

# ================================
# 15. RUN
# ================================
if __name__ == "__main__":
    query = "I earn 80k, expenses 30k, EMI 20k, savings 10k, want loan 10 lakh"
    result = app.invoke({"user_input": query})

    print("\n===== FINAL REPORT =====\n")
    print(result["final_report"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== FINAL REPORT =====

**Financial Health Report**

**Summary:**
This report provides an assessment of your current financial health, highlighting areas of strength and weakness. Based on the data provided, we have identified key recommendations to help you achieve a more stable financial foundation.

**Key Metrics:**

*   **Health:** Moderate
*   **Risk:** 50.0 (Moderate Risk Profile)
*   **Savings Ratio:** 12.5% (Below Ideal Target)
*   **EMI Ratio:** 25.0% (Above Ideal Target)

**Recommendations:**

1.  **Increase Savings Ratio**: Aim to increase your savings ratio to at least 30% of monthly income to achieve an ideal savings target.
2.  **Review EMI Ratio**: Ensure that your EMI ratio does not exceed 30% of monthly income and maintain a maximum Debt-to-Income (DTI) ratio of 60%.
3.  **Monitor Risk Profile**: Continue to monitor and manage your debt and savings to mitigate the moderate risk profile.
4.  **Implement Budgeting System**: Consider implementing a budgeting system to 